In [1]:
!pip install torch transformers tree_sitter==0.21.3 scikit-learn matplotlib -q

import os, json, random, copy, math, hashlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader, Dataset, Subset
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from transformers import (
    get_linear_schedule_with_warmup,
    RobertaConfig, RobertaModel,
    AutoTokenizer
)

from sklearn.metrics import (
    accuracy_score, f1_score,
    roc_auc_score, average_precision_score,
    classification_report
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import defaultdict

print("Imports OK")
print(f"CUDA: {torch.cuda.is_available()}")

# ─── CONFIGURATION ────────────────────────────────────────────────────────────
class Args:
    train_file         = "/kaggle/input/datasets/hasanmahmudabdullah/dfgdataset2/dataset_graphcodebert.jsonl"
    pretrained_encoder = "/kaggle/input/notebooks/iahmed223141/graphcodebert-train-text-only/saved_models/best_model_text_only.bin"

    model_name_or_path = "microsoft/graphcodebert-base"
    tokenizer_name     = "microsoft/graphcodebert-base"

    code_length        = 384
    train_batch_size   = 16
    eval_batch_size    = 32
    learning_rate      = 2e-5
    max_grad_norm      = 1.0
    num_train_epochs   = 5         # epochs per seed
    patience           = 2         # early stopping patience
    
    # ── KEY: list of seeds to run ──
    seeds              = [2025]

    # freeze_encoder = True  → only train classifier head
    # freeze_encoder = False → full fine-tune
    freeze_encoder     = False

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n_gpu  = torch.cuda.device_count()

args = Args()

print(f"Device  : {args.device}")
print(f"Seeds   : {args.seeds}")
print(f"Frozen  : {args.freeze_encoder}")

# ─── MODEL ────────────────────────────────────────────────────────────────────
class TextModel(nn.Module):
    def __init__(self, encoder, config, seed):
        super().__init__()
        self.encoder    = encoder
        self.config     = config
        self.dropout    = nn.Dropout(config.hidden_dropout_prob)

        # Initialise the head with a fixed seed so only the seed arg matters
        torch.manual_seed(seed)
        self.classifier = nn.Linear(config.hidden_size, 2)

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs[0]
        logits = self.classifier(self.dropout(sequence_output[:, 0, :]))
        prob = F.softmax(logits, dim=-1)

        if labels is not None:
            return CrossEntropyLoss()(logits, labels), prob
        return prob

# ─── DATASET ──────────────────────────────────────────────────────────────────
class SimpleCodeDataset(Dataset):
    def __init__(self, tokenizer, args, file_path):
        self.tokenizer = tokenizer
        self.args = args
        with open(file_path, 'r', encoding='utf-8') as f:
            self.lines = f.readlines()

    def __len__(self):
        return len(self.lines)

    def __getitem__(self, idx):
        entry = json.loads(self.lines[idx])
        code = entry.get('code', '')
        label = int(entry.get('label', 0)) if entry.get('label') is not None else 0
        tok = self.tokenizer(
            code, max_length=self.args.code_length,
            truncation=True, padding='max_length'
        )
        return {
            'input_ids': torch.tensor(tok['input_ids'], dtype=torch.long),
            'attention_mask': torch.tensor(tok['attention_mask'], dtype=torch.long),
            'label': torch.tensor(label, dtype=torch.long)
        }

# ─── STRATIFIED SPLIT ─────────────────────────────────────────────────────────
def infer_source(entry):
    for key in ("source", "dataset", "origin", "project"):
        value = entry.get(key)
        if value is not None and str(value).strip() != "":
            return str(value).strip()
    return "unknown"

def allocate_counts(total_needed, groups, fraction):
    raw = {g: len(v) * fraction for g, v in groups.items()}
    base = {g: int(math.floor(v)) for g, v in raw.items()}
    remainder = total_needed - sum(base.values())
    order = sorted(groups.keys(), key=lambda g: (raw[g] - base[g], len(groups[g])), reverse=True)
    for g in order[:remainder]:
        base[g] += 1
    return base

def get_stratified_indices(filepath, test_ratio=0.10, val_ratio=0.08, seed=42):
    # Streams instead of materialising all 199,960 entries (each carries a large
    # `dfg` array). Code hashes are collected in the same pass for the duplicate
    # filter applied after the split - see REMEDIATION_PLAN.md 5.1.
    srcs, hashes = [], []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            entry = json.loads(line)
            srcs.append(infer_source(entry))
            hashes.append(hashlib.md5(
                str(entry.get('code', '')).encode('utf-8', 'ignore')).hexdigest())
            del entry

    rng = random.Random(seed) # Fixed seed for splitting
    source_to_indices = defaultdict(list)
    for idx, s in enumerate(srcs):
        source_to_indices[s].append(idx)

    for indices in source_to_indices.values():
        rng.shuffle(indices)

    total = len(srcs)
    target_test = int(round(total * test_ratio))
    target_val = int(round(total * val_ratio))
    target_train = total - target_test - target_val

    test_alloc = allocate_counts(target_test, source_to_indices, test_ratio)
    trainval_groups = {}
    test_indices = []
    for source, indices in source_to_indices.items():
        take = min(test_alloc[source], len(indices))
        test_indices.extend(indices[:take])
        trainval_groups[source] = indices[take:]

    adjusted_val_ratio = val_ratio / (1.0 - test_ratio)
    val_alloc = allocate_counts(target_val, trainval_groups, adjusted_val_ratio)

    val_indices, train_indices = [], []
    for source, indices in trainval_groups.items():
        take = min(val_alloc[source], len(indices))
        val_indices.extend(indices[:take])
        train_indices.extend(indices[take:])

    return sorted(train_indices), sorted(val_indices), sorted(test_indices), hashes

print("Calculating stratified split (82/8/10) with fixed seed=42...")
train_indices, val_indices, test_indices, code_hashes = get_stratified_indices(args.train_file)

# Duplicate filter: drop test entries byte-identical to a train/val entry so the
# model is scored only on code it never saw. Training is unaffected.
_seen = {code_hashes[i] for i in train_indices}
_seen.update(code_hashes[i] for i in val_indices)
_before = len(test_indices)
test_indices = [i for i in test_indices if code_hashes[i] not in _seen]
print(f"Duplicate filter: dropped {_before - len(test_indices):,} "
      f"({(_before - len(test_indices))/_before:.2%}) -> {len(test_indices):,} clean")
print(f"Train: {len(train_indices)}, Val: {len(val_indices)}, Test: {len(test_indices)}")

# ─── EVALUATION FUNCTION ──────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, dataset, desc="Eval"):
    loader = DataLoader(dataset, batch_size=args.eval_batch_size, num_workers=2, pin_memory=True)
    model.eval()
    preds, labels, probs_list = [], [], []
    for batch in loader:
        inp = {
            'input_ids': batch['input_ids'].to(args.device),
            'attention_mask': batch['attention_mask'].to(args.device)
        }
        prob = model(**inp)
        probs_list.extend(prob[:, 1].cpu().numpy())
        preds.extend(torch.argmax(prob, dim=-1).cpu().numpy())
        labels.extend(batch['label'].numpy())
    model.train()
    
    acc = accuracy_score(labels, preds)
    roc = roc_auc_score(labels, probs_list)
    return acc, roc, preds, np.array(probs_list), np.array(labels)

# ─── TRAINING LOOP ───────────────────────────────────────────────────────────
def train_one_seed(model, train_ds, val_ds, seed):
    loader = DataLoader(
        train_ds, batch_size=args.train_batch_size,
        shuffle=True, num_workers=2, pin_memory=True
    )

    params = model.classifier.parameters() if args.freeze_encoder else model.parameters()
    optimizer = AdamW(params, lr=args.learning_rate, eps=1e-8)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=0,
        num_training_steps=len(loader) * args.num_train_epochs
    )
    scaler = GradScaler('cuda', enabled=torch.cuda.is_available())

    best_val_acc = -1.0
    patience_counter = 0
    best_model_state = None

    for epoch in range(args.num_train_epochs):
        model.train()
        tr_loss = 0.0
        for batch in tqdm(loader, desc=f"   [seed {seed}] Epoch {epoch}", disable=True):
            inp = {
                'input_ids': batch['input_ids'].to(args.device),
                'attention_mask': batch['attention_mask'].to(args.device),
                'labels': batch['label'].to(args.device)
            }
            optimizer.zero_grad()
            with autocast('cuda'):
                loss, _ = model(**inp)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            tr_loss += loss.item()

        avg_loss = tr_loss / len(loader)
        
        # Validation
        val_acc, val_roc, _, _, _ = evaluate(model, val_ds, desc="Validation")
        print(f"   Epoch {epoch} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4%} | Val ROC: {val_roc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            best_model_state = copy.deepcopy(model.state_dict())
            print(f"   * New best validation accuracy!")
        else:
            patience_counter += 1
            print(f"   * No improvement. Patience {patience_counter}/{args.patience}")
            if patience_counter >= args.patience:
                print("   * Early stopping triggered.")
                break

    # Load best state
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    return model

# ─── MAIN: MULTI-SEED LOOP ────────────────────────────────────────────────────
if not args.pretrained_encoder or not os.path.exists(args.pretrained_encoder):
    print(f"Please specify a valid pretrained_encoder path. Given: {args.pretrained_encoder}")
else:
    print("Loading shared components...")
    config    = RobertaConfig.from_pretrained(args.model_name_or_path)
    config.num_labels = 2
    tokenizer = AutoTokenizer.from_pretrained(args.tokenizer_name, use_fast=True)
    print("  ✓ Config & tokenizer ready")

    print("Loading dataset...")
    full_ds  = SimpleCodeDataset(tokenizer, args, args.train_file)
    train_ds = Subset(full_ds, train_indices)
    val_ds   = Subset(full_ds, val_indices)
    test_ds  = Subset(full_ds, test_indices)

    # Load the pre-trained encoder weights ONCE
    print("\nLoading pre-trained encoder...")
    encoder_base = RobertaModel.from_pretrained(args.model_name_or_path, config=config)
    pretrained   = torch.load(args.pretrained_encoder, map_location='cpu')
    
    # Extract only encoder weights (strip classifier keys if present)
    encoder_state = {k.replace('encoder.', ''): v
                     for k, v in pretrained.items() if k.startswith('encoder.') or not k.startswith('classifier.')}
    encoder_base.load_state_dict(encoder_state, strict=False)
    print("  ✓ Encoder weights loaded")

    seed_results = []

    for seed in args.seeds:
        print(f"\n{'─'*55}")
        print(f"SEED {seed}")
        print(f"{'─'*55}")

        # Fix all RNG sources for this seed
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if args.n_gpu > 0:
            torch.cuda.manual_seed_all(seed)

        # Deep-copy encoder so each seed starts from identical pretrained weights
        enc   = copy.deepcopy(encoder_base)
        model = TextModel(enc, config, seed).to(args.device)

        if args.freeze_encoder:
            for param in model.encoder.parameters():
                param.requires_grad = False
            trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"  Encoder frozen. Trainable params: {trainable:,}")

        model = train_one_seed(model, train_ds, val_ds, seed)

        # Evaluate once on the held-out test set
        test_acc, _, test_preds, test_probs, test_labels = evaluate(model, test_ds, desc="Final Test")

        roc_auc = roc_auc_score(test_labels, test_probs)
        pr_auc  = average_precision_score(test_labels, test_probs)
        f1_mac  = f1_score(test_labels, test_preds, average='macro')
        f1_mal  = f1_score(test_labels, test_preds, pos_label=1)

        seed_results.append({
            'seed':    seed,
            'acc':     test_acc,
            'roc_auc': roc_auc,
            'pr_auc':  pr_auc,
            'f1_mac':  f1_mac,
            'f1_mal':  f1_mal
        })

        print(f"\n  Seed {seed} → Acc={test_acc:.4%}  ROC={roc_auc:.4f}  PR={pr_auc:.4f}  F1={f1_mac:.4f}\n")

    print("All seeds done!")

# ─── AGGREGATE & SAVE RESULTS ─────────────────────────────────────────────────
if 'seed_results' in locals() and seed_results:
    accs = [r['acc'] for r in seed_results]
    rocs = [r['roc_auc'] for r in seed_results]
    prs  = [r['pr_auc'] for r in seed_results]
    f1s  = [r['f1_mac'] for r in seed_results]

    m_acc, s_acc = np.mean(accs), np.std(accs)
    m_roc, s_roc = np.mean(rocs), np.std(rocs)
    m_pr,  s_pr  = np.mean(prs),  np.std(prs)
    m_f1,  s_f1  = np.mean(f1s),  np.std(f1s)

    report = (
        f"=== Test 3: Multi-Seed Robustness ({len(args.seeds)} seeds) ===\n"
        f"Accuracy  : {m_acc:.2%} ± {s_acc:.2%}\n"
        f"ROC-AUC   : {m_roc:.4f} ± {s_roc:.4f}\n"
        f"PR-AUC    : {m_pr:.4f} ± {s_pr:.4f}\n"
        f"F1 (macro): {m_f1:.4f} ± {s_f1:.4f}\n"
        "\nPer-seed details:\n"
    )
    for r in seed_results:
        report += f"Seed {r['seed']:>4}: Acc={r['acc']:.4%} ROC={r['roc_auc']:.4f} PR={r['pr_auc']:.4f}\n"

    print(report)

    os.makedirs('/kaggle/working', exist_ok=True)
    with open('/kaggle/working/test3_seed2025_results.txt', 'w') as f:
        f.write(report)
    print("Saved summary to /kaggle/working/test3_seed2025_results.txt")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.2/502.2 kB 9.4 MB/s eta 0:00:00
Imports OK
CUDA: True
Device  : cuda
Seeds   : [2025]
Frozen  : False
Calculating stratified split (82/8/10) with fixed seed=42...
Train: 163967, Val: 15997, Test: 19996
Loading shared components...


config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

   Epoch 2 | Loss: 0.2054 | Val Acc: 88.2165% | Val ROC: 0.9597
   * New best validation accuracy!


/tmp/ipykernel_23/4130376124.py:218: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


   Epoch 3 | Loss: 0.1670 | Val Acc: 88.6479% | Val ROC: 0.9581
   * New best validation accuracy!


/tmp/ipykernel_23/4130376124.py:218: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


   Epoch 4 | Loss: 0.1398 | Val Acc: 88.2540% | Val ROC: 0.9547
   * No improvement. Patience 1/2

  Seed 2025 → Acc=89.0728%  ROC=0.9596  PR=0.9617  F1=0.8907

All seeds done!
=== Test 3: Multi-Seed Robustness (1 seeds) ===
Accuracy  : 89.07% ± 0.00%
ROC-AUC   : 0.9596 ± 0.0000
PR-AUC    : 0.9617 ± 0.0000
F1 (macro): 0.8907 ± 0.0000

Per-seed details:
Seed 2025: Acc=89.0728% ROC=0.9596 PR=0.9617

Saved summary to /kaggle/working/test3_seed2025_results.txt
